In [13]:
# setting the environment variables, the keys
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from config import set_environment

# for the keys - as explained early in chapter 2
set_environment()

In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Custom tools

We will use `numexpr` library to evaluate math expressions:

In [34]:
import numexpr as ne
import math

math_constants = {
    "pi": math.pi,
    "i": 1j,
    "e": math.exp
}
c = ne.evaluate(("2+2"), local_dict=math_constants)
c

array(4, dtype=int32)

In [35]:
ne.evaluate("(2+3*i)**2", local_dict=math_constants)

array(-5.+12.j)

We define our calculator as a Python function and wrap it with a built-in `@tool` decorator to create a tool from it:

In [39]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """Calculates a single mathematical expression, incl. complex numbers

    Always add * operations, examples:
        73i -> 73*i
        7pi**2 -> 7*pi**2
    """
    math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
    result = ne.evaluate(expression.strip(), local_dict=math_constants)
    return str(result)

In [40]:
from langchain_core.tools import BaseTool

assert isinstance(calculator, BaseTool)
print(f"Tool name: {calculator.name}")
print(f"Tool description: {calculator.description}")
print(f"Tool schema: {calculator.args_schema.model_json_schema()}")

Tool name: calculator
Tool description: Calculates a single mathematical expression, incl. complex numbers

    Always add * operations, examples:
        73i -> 73*i
        7pi**2 -> 7*pi**2
Tool schema: {'description': 'Calculates a single mathematical expression, incl. complex numbers\n\nAlways add * operations, examples:\n    73i -> 73*i\n    7pi**2 -> 7*pi**2', 'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}


In [41]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
query = "How much is 2+3i squared?"

agent = create_react_agent(llm, [calculator])

for event in agent.stream({"messages": [HumanMessage(query)]}, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How much is 2+3i squared?
================================== Ai Message ==================================
Tool Calls:
  calculator (026b78c6-3ca1-4505-86c6-bc9ed1c592f2)
 Call ID: 026b78c6-3ca1-4505-86c6-bc9ed1c592f2
  Args:
    expression: (2+3i)**2
================================= Tool Message =================================
Name: calculator

Error: SyntaxError('invalid decimal literal', ('<expr>', 1, 4, '(2+3i)**2', 1, 4))
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  calculator (67b293db-71ff-449f-ae73-f16d19cdeb4d)
 Call ID: 67b293db-71ff-449f-ae73-f16d19cdeb4d
  Args:
    expression: (2 + 3*i)**2
================================= Tool Message =================================
Name: calculator

(-5+12j)
================================== Ai Message ==================================

(2 + 3i) squared is -5 + 12i.


We will re-use the `search` tool we created in the previous section:

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

In [42]:
from langchain_core.messages import SystemMessage

query = "What is a square root of the current US president's age multiplied by 132?"
system_prompt = SystemMessage(
    "Think step-by-step. Always use search to get the fresh information about events or public facts that can change over time. Now is 2025 and remember president elections in the US recently happened."
)

agent = create_react_agent(llm, [calculator, search], prompt=system_prompt)

for event in agent.stream({"messages": [HumanMessage(query)]}, stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is a square root of the current US president's age multiplied by 132?
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (5caa34b2-a575-4583-8291-e91ff64ae229)
 Call ID: 5caa34b2-a575-4583-8291-e91ff64ae229
  Args:
    query: current US president age 2025
================================= Tool Message =================================
Name: duckduckgo_search

2 weeks ago - The first table charts the age of each president of the United States at the time of their inauguration (first inauguration if elected to multiple and consecutive terms), upon leaving office, and at the time of death. Presidents who are still living have their lifespans and post-presidency timespans calculated through December 22, 2025... 8 hours ago - Trump won the election in November 2024 with 312 electoral votes to incumbent vice president Kamala Harris's 226. He als

We can explore the final answer the model produced:

In [43]:
print(event["messages"][-1].content)

Donald Trump is the current US President, born on June 14, 1946. As of 2025, he is 78 years old. The square root of his age multiplied by 132 is approximately 1165.79.


We can also create a tool from a `Runnable` (and again, we can convery any Python function to a `Runnable` by using `RunnableLambda`):

In [44]:
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import convert_runnable_to_tool


def calculator(expression: str) -> str:
    math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
    result = ne.evaluate(expression.strip(), local_dict=math_constants)
    return str(result)

calculator_with_retry = RunnableLambda(calculator).with_retry(
    wait_exponential_jitter=True,
    stop_after_attempt=3,
)

calculator_tool = convert_runnable_to_tool(
    calculator_with_retry,
    name="calculator",
    description=(
        "Calculates a single mathematical expression, incl. complex numbers."
        "'\nAlways add * to operations, examples:\n73i -> 73*i\n"
        "7pi**2 -> 7*pi**2"
    ),
    arg_types={"expression": "str"}
)

In [45]:
calculator_tool.invoke({"expression": "(2+3*i)**2"})

'(-5+12j)'

In [59]:
llm.invoke("How much is (2+3i)**2", tools=[calculator_tool]).tool_calls[0]

{'name': 'calculator',
 'args': {'__arg1': '(2+3*i)**2'},
 'id': '566d8cd8-5aba-4ee2-909e-1b830a395673',
 'type': 'tool_call'}

We can also pass a custom arguments schema when we create a tool with `convert_runnable_to_tool`:

In [71]:
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableConfig

class CalculatorArgs(BaseModel):
    expression: str = Field(description="Mathematical expression to be evaluated")

def calculator(state: CalculatorArgs, config: RunnableConfig) -> str:
    expression = state["expression"]
    math_constants = config["configurable"].get("math_constants", {})
    result = ne.evaluate(expression.strip(), local_dict=math_constants)
    return str(result)

calculator_with_retry = RunnableLambda(calculator).with_retry(
    wait_exponential_jitter=True,
    stop_after_attempt=3,
)

calculator_tool = convert_runnable_to_tool(
    calculator_with_retry,
    name="calculator",
    description=(
        "Calculates a single mathematical expression, incl. complex numbers."
        "'\nAlways add * to operations, examples:\n73i -> 73*i\n"
        "7pi**2 -> 7*pi**2"
    ),
    args_schema=CalculatorArgs,
    arg_types={"expression": "str"},
)

In [72]:
assert isinstance(calculator_tool, BaseTool)
print(f"Tool name: {calculator_tool.name}")
print(f"Tool description: {calculator_tool.description}")
print(f"Args schema: {calculator_tool.args_schema.model_json_schema()}")

Tool name: calculator
Tool description: Calculates a single mathematical expression, incl. complex numbers.'
Always add * to operations, examples:
73i -> 73*i
7pi**2 -> 7*pi**2
Args schema: {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculator', 'type': 'object'}


That's how we pass configuration to our new tool:

In [73]:
math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
config = {"configurable": {"math_constants": math_constants}}

tool_call = llm.invoke("How much is (2+3i)**2", tools=[calculator_tool]).tool_calls[0]
print(tool_call)

{'name': 'calculator', 'args': {'expression': '(2+3*i)**2'}, 'id': '7d3c1668-4a12-49d9-ae85-d647e2b80e55', 'type': 'tool_call'}


In [ ]:
calculator_tool.invoke(tool_call["args"], config=config)

'(-5+12j)'

In [76]:
from langchain_core.tools import StructuredTool


def calculator(expression: str) -> str:
    math_constants = {"pi": math.pi, "i": 1j, "e": math.exp}
    result = ne.evaluate(expression.strip(), local_dict=math_constants)
    return str(result)

calculator_tool = StructuredTool.from_function(
    name="calculator",
    description=(
        "Calculates a single mathematical expression, incl. complex numbers."
    ),
    func=calculator,
    args_schema=CalculatorArgs,
)

tool_call = llm.invoke("How much is (2+3i)**2", tools=[calculator_tool]).tool_calls[0]
print(tool_call)

{'name': 'calculator', 'args': {'expression': '(2+3i)**2'}, 'id': 'db398465-ffaa-415e-99b2-135299bc5e57', 'type': 'tool_call'}


We can also take a look at how our agents adapts to the feedback from the environment and how an LLM corrects the input sent to the tool:

In [78]:
from langchain_core.tools import StructuredTool

def calculator(expression: str) -> str:
    """Calculates a single mathematical expression, incl. complex numbers."""
    return str(ne.evaluate(expression.strip(), local_dict={}))

calculator_tool = StructuredTool.from_function(
    func=calculator,
    handle_tool_error=True,
)

agent = create_react_agent(
    llm, [calculator_tool]
)

for event in agent.stream({"messages": [HumanMessage("How much is (2+3i)^2")]}, stream_mode="values", config=config):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

How much is (2+3i)^2
================================== Ai Message ==================================
Tool Calls:
  calculator (511ddaa9-32d9-4c9d-96c0-58b6092e9067)
 Call ID: 511ddaa9-32d9-4c9d-96c0-58b6092e9067
  Args:
    expression: (2+3i)^2
================================= Tool Message =================================
Name: calculator

Error: SyntaxError('invalid decimal literal', ('<expr>', 1, 4, '(2+3i)^2', 1, 4))
 Please fix your mistakes.
================================== Ai Message ==================================

It seems there was a problem with the way I formatted the complex number. I'll try again using 'j' for the imaginary unit instead of 'i'.
Tool Calls:
  calculator (5ab6d05d-fdb7-4fd6-bba9-39de024d2db7)
 Call ID: 5ab6d05d-fdb7-4fd6-bba9-39de024d2db7
  Args:
    expression: (2+3j)^2
================================= Tool Message =================================
Name: calculator

E